## new har gene list inspection

In [102]:
import scanpy as sc
import pandas as pd
import numpy as np

In [4]:
har700=pd.read_csv("/scratch200/reutj/data/hsg_gene_lists/har700.csv")

In [5]:
har700

,Chr,Start,End,Func.refGene,Gene.refGene,HAR,HAR.1,ChromHMM-Epigenomics Roadmap,"Interacting genes (Goh et al., 2012; Jin et al., 2013; Li et al., 2010; Li et al., 2012; Ma et al., 2015)",Unnamed: 9
0,chr1,8060033,8060380,intergenic,"PARK7,ERRFI1",Bird_HAR165,HAR_Merge50-00005,15_Quies-Blood_&_T-cell-CD8_Naive_Primary_Cell...,ERRFI1,NaN
1,chr1,19688085,19688407,intronic,CAPZB,"Toh_HAR234,Bird_HAR74",HAR_Merge50-00009,"7_Enh-Brain-Brain_Mid_Frontal_Lobe,5_TxWk-Othe...",NBL1,NaN
2,chr1,20713360,20713539,ncRNA_intronic,LOC339505,Bird_HAR452,HAR_Merge50-00010,"7_Enh-Brain-Brain_Mid_Frontal_Lobe,13_ReprPC-E...","DDOST,MUL1,PINK1,LOC339505",NaN
3,chr1,25887868,25887885,intronic,LDLRAP1,Toh_HAR355,HAR_Merge50-00012,"7_Enh-Brain-Brain_Mid_Frontal_Lobe,5_TxWk-Hear...","MAN1C1,FAM54B,LDLRAP1,SEPN1",NaN
4,chr1,27309617,27309945,intergenic,"C1orf172,TRNP1",Bird_HAR414,HAR_Merge50-00013,15_Quies-Blood_&_T-cell-CD8_Naive_Primary_Cell...,"TRNP1,FAM46B,SLC9A1",NaN
...,...,...,...,...,...,...,...,...,...,...
2732,chrX,147328163,147328181,intergenic,"FMR1NB,AFF2",Toh_HAR139,HAR_Merge50-02733,15_Quies-ES-deriv-hESC_Derived_CD56+_Mesoderm_...,NaN,NaN
2733,chrX,147783516,147783714,intronic,AFF2,Toh_HAR315,HAR_Merge50-02734,15_Quies-ES-deriv-hESC_Derived_CD56+_Mesoderm_...,NaN,NaN
2734,chrX,148836615,148836733,intergenic,"MAGEA11,HSFX1",Bird_HAR20,HAR_Merge50-02735,15_Quies-ES-deriv-hESC_Derived_CD56+_Mesoderm_...,NaN,NaN
2735,chrX,149314360,149314608,intergenic,"LINC00894,MIR2114",Bird_HAR1330,HAR_Merge50-02736,15_Quies-ES-deriv-hESC_Derived_CD56+_Mesoderm_...,NaN,NaN


In [6]:
har700=har700.rename(columns={"Func.refGene":"location","Gene.refGene":"refGene","Interacting genes (Goh et al., 2012; Jin et al., 2013; Li et al., 2010; Li et al., 2012; Ma et al., 2015)":"interacting_genes"})

In [7]:
all_genes=list(map(lambda x: x.split(","),har700["refGene"].tolist()))
genes_2000=[]
for i in all_genes:
    for j in i:
        if j not in genes_2000:
            genes_2000.append(j)
print(len(genes_2000))

2163


In [8]:
interact_genes=list(map(lambda x: x.split(","),har700.loc[:566,"interacting_genes"].tolist()))
#from 566 all intercating are none this is why i go to line 566 only 
genes_700=[]
for i in interact_genes:
    for j in i:
        if j not in genes_700:
            genes_700.append(j)
print(len(genes_700))

830


In [ ]:
#save them to gmt - DONE- find where the code is and chnge so it fits here..

#update custom_har_genes gmt list
output_file="/scratch200/reutj/data/hsg_gene_lists/custom_har_genes_list.gmt"
with open(output_file, 'w') as f:
    # Create a line in GMT format for one gene set
    line = f"{har_genes}\tDescription\t{'\t'.join(har_gene_symb)}\n"
    f.write(line)

In [ ]:
#extract ensg for all..

In [9]:
import mygene

mg = mygene.MyGeneInfo()

In [50]:
res=pd.DataFrame(mg.querymany(genes_700, scopes='symbol', fields='ensembl.gene', species='human'))
res

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
12 input query terms found dup hits:	[('SLC25A3P1', 2), ('7SK', 10), ('LINC00710', 2), ('GOLGA2P6', 2), ('BMS1P4', 2), ('LINC00608', 2), 
153 input query terms found no hit:	['LOC339505', 'FAM54B', 'SEPN1', 'FAM46B', 'CCDC23', 'FLJ32224', 'LOC400752', 'AK097571', 'PPAP2B', 


,query,_id,_score,ensembl,notfound
0,ERRFI1,54206,18.223173,{'gene': 'ENSG00000116285'},NaN
1,NBL1,4681,17.867332,{'gene': 'ENSG00000158747'},NaN
2,DDOST,1650,17.590942,{'gene': 'ENSG00000244038'},NaN
3,MUL1,79594,17.173996,{'gene': 'ENSG00000090432'},NaN
4,PINK1,65018,17.152830,{'gene': 'ENSG00000158828'},NaN
...,...,...,...,...,...
853,CHST7,56548,17.688334,{'gene': 'ENSG00000147119'},NaN
854,GRIPAP1,56850,17.867332,"[{'gene': 'ENSG00000068400'}, {'gene': 'ENSG00...",NaN
855,PRPS1,5631,18.393486,{'gene': 'ENSG00000147224'},NaN
856,LHFPL1,340596,18.950712,{'gene': 'ENSG00000182508'},NaN


In [51]:
#function to gather the hits for each gene together 
har700_dict={}
for i in range(len(res)):
    gene=res['query'][i]
    har700_dict[gene]=[]
for i in range(len(res)):
    gene=res['query'][i]
    ens=res.ensembl[i]
    if type(ens)==dict:
        har700_dict[gene].append(ens['gene'])
    elif type(ens)==list:
        har700_dict[gene]=ens

har700_dict

{'ERRFI1': ['ENSG00000116285'],
 'NBL1': ['ENSG00000158747'],
 'DDOST': ['ENSG00000244038'],
 'MUL1': ['ENSG00000090432'],
 'PINK1': ['ENSG00000158828'],
 'LOC339505': [],
 'MAN1C1': ['ENSG00000117643'],
 'FAM54B': [],
 'LDLRAP1': ['ENSG00000157978'],
 'SEPN1': [],
 'TRNP1': ['ENSG00000253368'],
 'FAM46B': [],
 'SLC9A1': ['ENSG00000090020'],
 'INPP5B': ['ENSG00000204084'],
 'CCDC23': [],
 'FLJ32224': [],
 'SLC2A1': ['ENSG00000117394'],
 'DMAP1': ['ENSG00000178028'],
 'RNF220': ['ENSG00000187147'],
 'TMEM53': ['ENSG00000126106'],
 'PTCH2': ['ENSG00000117425'],
 'LOC400752': [],
 'PLK3': ['ENSG00000173846'],
 'BTBD19': ['ENSG00000222009'],
 'AKR1A1': ['ENSG00000117448'],
 'TOE1': ['ENSG00000132773'],
 'EIF2B3': ['ENSG00000070785'],
 'MUTYH': ['ENSG00000132781'],
 'SPATA6': ['ENSG00000132122'],
 'CDKN2C': ['ENSG00000123080'],
 'AK097571': [],
 'SLC25A3P1': ['ENSG00000293595', 'ENSG00000236253'],
 'PPAP2B': [],
 'NFIA': ['ENSG00000162599'],
 'LMO4': ['ENSG00000143013'],
 'LOC339524': [],
 

In [ ]:
#extract genes wit multiple matchings..
SLC25A3P1(2)
7SK-ENSG00000283293
BMS1P4(1)
APEX1(1)
OSGEP(2)
BTBD7(2)
C16orf46(2)
DDX52(1)
SYNRG(1)
ARHGAP23(2)
DSEL(2)
CRIM1(2)
HAT1(2)
DYNC1I2(2)
INO80D(1)
RBFOX2(1)
ELFN2(1)
MASP1(2)
U6- LOT OF GENES , NO ONE ENSEMBL
CASP8AP2(1)
COG5(1)
DLC1(2)
GSDMD(2)
ZC3H3(2)
GRIPAP1(1)

In [52]:
#present as a dataframe
har700_dict=pd.DataFrame.from_dict(har700_dict,orient='index').iloc[:,:1]
har700_dict.loc[har700_dict.index=="SLC25A3P1",0]='ENSG00000236253'
har700_dict.loc[har700_dict.index=="7SK",0]='ENSG00000283293'
har700_dict.loc[har700_dict.index=="BMS1P4",0]='ENSG00000242338'
har700_dict.loc[har700_dict.index=="APEX1",0]='ENSG00000100823'
har700_dict.loc[har700_dict.index=="OSGEP",0]='ENSG00000092094'
har700_dict.loc[har700_dict.index=="BTBD7",0]='ENSG00000011114'
har700_dict.loc[har700_dict.index=="C16orf46",0]='ENSG00000166455'
har700_dict.loc[har700_dict.index=="DDX52",0]='ENSG00000278053'
har700_dict.loc[har700_dict.index=="SYNRG",0]='ENSG00000275066'
har700_dict.loc[har700_dict.index=="ARHGAP23",0]='ENSG00000275832'
har700_dict.loc[har700_dict.index=="DSEL",0]='ENSG00000171451'
har700_dict.loc[har700_dict.index=="CRIM1",0]='ENSG00000150938'
har700_dict.loc[har700_dict.index=="HAT1",0]='ENSG00000128708'
har700_dict.loc[har700_dict.index=="DYNC1I2",0]='ENSG00000077380'
har700_dict.loc[har700_dict.index=="INO80D",0]='ENSG00000114933'
har700_dict.loc[har700_dict.index=="RBFOX2",0]='ENSG00000100320'
har700_dict.loc[har700_dict.index=="ELFN2",0]='ENSG00000166897'
har700_dict.loc[har700_dict.index=="MASP1",0]='ENSG00000127241'
har700_dict.loc[har700_dict.index=="CASP8AP2",0]='ENSG00000118412'
har700_dict.loc[har700_dict.index=="COG5",0]='ENSG00000164597'
har700_dict.loc[har700_dict.index=="DLC1",0]='ENSG00000164741'
har700_dict.loc[har700_dict.index=="GSDMD",0]='ENSG00000104518'
har700_dict.loc[har700_dict.index=="ZC3H3",0]='ENSG00000014164'
har700_dict.loc[har700_dict.index=="GRIPAP1",0]='ENSG00000068400'
har700_dict.loc[har700_dict.index=="U6",0]= None

In [53]:
har700_dict=har700_dict.rename(columns={0:'ens'})
har700_dict

,ens
ERRFI1,ENSG00000116285
NBL1,ENSG00000158747
DDOST,ENSG00000244038
MUL1,ENSG00000090432
PINK1,ENSG00000158828
...,...
CHST7,ENSG00000147119
GRIPAP1,ENSG00000068400
PRPS1,ENSG00000147224
LHFPL1,ENSG00000182508


In [55]:
har700_df=pd.DataFrame(data={'gene':har700_dict.index,'ens':har700_dict.ens})
har700_df.index=range(830)
har700_df=har700_df[~har700_df.ens.isna()]
har700_df

,gene,ens
0,ERRFI1,ENSG00000116285
1,NBL1,ENSG00000158747
2,DDOST,ENSG00000244038
3,MUL1,ENSG00000090432
4,PINK1,ENSG00000158828
...,...,...
825,CHST7,ENSG00000147119
826,GRIPAP1,ENSG00000068400
827,PRPS1,ENSG00000147224
828,LHFPL1,ENSG00000182508


In [56]:
har700_df.to_csv("/scratch200/reutj/data/hsg_gene_lists/har700_df.csv")

In [57]:
#do the same with interacting genes:
res=pd.DataFrame(mg.querymany(genes_2000, scopes='symbol', fields='ensembl.gene', species='human'))
res

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
62 input query terms found dup hits:	[('TSNAX-DISC1', 2), ('LINC00710', 2), ('LINC00900', 2), ('NR2F2-AS1', 2), ('YBX3P1', 2), ('CLUHP3',
259 input query terms found no hit:	['LOC339505', 'C1orf172', 'CCDC23', 'PPAP2B', 'LOC100505768', 'FLJ31662', 'NONE', 'LOC646268', 'HIST


,query,_id,_score,ensembl,notfound
0,PARK7,11315,17.616903,{'gene': 'ENSG00000116288'},NaN
1,ERRFI1,54206,18.223173,{'gene': 'ENSG00000116285'},NaN
2,CAPZB,832,17.542368,{'gene': 'ENSG00000077549'},NaN
3,LOC339505,NaN,NaN,NaN,True
4,LDLRAP1,26119,18.178656,{'gene': 'ENSG00000157978'},NaN
...,...,...,...,...,...
2223,MAGEA11,4110,22.968216,{'gene': 'ENSG00000185247'},NaN
2224,HSFX1,100506164,24.671068,{'gene': 'ENSG00000171116'},NaN
2225,LINC00894,NaN,NaN,NaN,True
2226,MIR2114,100313839,24.477064,{'gene': 'ENSG00000252454'},NaN


In [64]:
#function to gather the hits for each gene together 
har2000_dict={}
for i in range(len(res)):
    gene=res['query'][i]
    har2000_dict[gene]=[]
for i in range(len(res)):
    gene=res['query'][i]
    ens=res.ensembl[i]
    if type(ens)==dict:
        har2000_dict[gene].append(ens['gene'])
    elif type(ens)==list:
        har2000_dict[gene]=ens

In [59]:
#find all duplicates
[(i,j) for i,j in har2000_dict.items() if len(j)>1]

[('C14orf180', [{'gene': 'ENSG00000184601'}, {'gene': 'ENSG00000274126'}]),
 ('MEGF11', [{'gene': 'ENSG00000157890'}, {'gene': 'ENSG00000277848'}]),
 ('CLUHP3', ['ENSG00000290927', 'ENSG00000131797']),
 ('VPS53', [{'gene': 'ENSG00000141252'}, {'gene': 'ENSG00000283883'}]),
 ('AATF', [{'gene': 'ENSG00000276072'}, {'gene': 'ENSG00000275700'}]),
 ('SLC39A11', [{'gene': 'ENSG00000282291'}, {'gene': 'ENSG00000133195'}]),
 ('SALL3',
  [{'gene': 'ENSG00000256463'},
   {'gene': 'ENSG00000263310'},
   {'gene': 'ENSG00000277015'}]),
 ('CRIM1', [{'gene': 'ENSG00000277354'}, {'gene': 'ENSG00000150938'}]),
 ('DYNC1I2', [{'gene': 'ENSG00000292102'}, {'gene': 'ENSG00000077380'}]),
 ('INO80D', [{'gene': 'ENSG00000114933'}, {'gene': 'ENSG00000283510'}]),
 ('RBFOX2', [{'gene': 'ENSG00000100320'}, {'gene': 'ENSG00000277564'}]),
 ('LINC01060', ['ENSG00000281580', 'ENSG00000249378']),
 ('DLC1', [{'gene': 'ENSG00000288673'}, {'gene': 'ENSG00000164741'}]),
 ('GSDMC', [{'gene': 'ENSG00000147697'}, {'gene': 'E

In [65]:
#extract genes wit multiple matchings..
har2000_dict=pd.DataFrame.from_dict(har2000_dict,orient='index').iloc[:,:1]
har2000_dict.loc[har2000_dict.index=="GTF2IRD1P1",0]='ENSG00000230583'
har2000_dict.loc[har2000_dict.index=="MYADML",0]='ENSG00000239649'
har2000_dict.loc[har2000_dict.index=="CEP170",0]='ENSG00000143702'
har2000_dict.loc[har2000_dict.index=="HHAT",0]='ENSG00000054392'
har2000_dict.loc[har2000_dict.index=="SERTAD4",0]='ENSG00000082497'
har2000_dict.loc[har2000_dict.index=="KCNT2",0]='ENSG00000162687'
har2000_dict.loc[har2000_dict.index=="LINC00869",0]='ENSG00000277147'
har2000_dict.loc[har2000_dict.index=="ASAP3",0]='ENSG00000088280'
har2000_dict.loc[har2000_dict.index=="TTC34",0]='ENSG00000215912'
har2000_dict.loc[har2000_dict.index=="WDR45",0]='ENSG00000196998'
har2000_dict.loc[har2000_dict.index=="LINC01060",0]='ENSG00000249378'
har2000_dict.loc[har2000_dict.index=="GSDMC",0]='ENSG00000147697'
har2000_dict.loc[har2000_dict.index=="SALL3",0]='ENSG00000256463'
har2000_dict.loc[har2000_dict.index=="SLC39A11",0]='ENSG00000133195'
har2000_dict.loc[har2000_dict.index=="AATF",0]='ENSG00000275700'
har2000_dict.loc[har2000_dict.index=="VPS53",0]='ENSG00000141252'
har2000_dict.loc[har2000_dict.index=="CLUHP3",0]='ENSG00000131797'
har2000_dict.loc[har2000_dict.index=="MEGF11",0]='ENSG00000157890'
har2000_dict.loc[har2000_dict.index=="C14orf180",0]='ENSG00000184601'
har2000_dict.loc[har2000_dict.index=="UBE2NL",0]='ENSG00000276380'
har2000_dict.loc[har2000_dict.index=="GLUD2",0]='ENSG00000182890'
har2000_dict.loc[har2000_dict.index=="MROH5",0]='ENSG00000226807'
har2000_dict.loc[har2000_dict.index=="PRSS55",0]='ENSG00000184647'
har2000_dict.loc[har2000_dict.index=="MSRA",0]='ENSG00000175806'
har2000_dict.loc[har2000_dict.index=="PPP1R3B",0]='ENSG00000173281'
har2000_dict.loc[har2000_dict.index=="CNTNAP2",0]='ENSG00000174469'
har2000_dict.loc[har2000_dict.index=="DPY19L2P4",0]='ENSG00000235436'
har2000_dict.loc[har2000_dict.index=="STAG3L4",0]='ENSG00000106610'
har2000_dict.loc[har2000_dict.index=="SDCCAG8",0]='ENSG00000054282'
har2000_dict.loc[har2000_dict.index=="KIF26B",0]='ENSG00000162849'
har2000_dict.loc[har2000_dict.index=="RNH1",0]='ENSG00000023191'
har2000_dict.loc[har2000_dict.index=="OR4C12",0]='ENSG00000221954'
har2000_dict.loc[har2000_dict.index=="OR8K1",0]='ENSG00000150261'
har2000_dict.loc[har2000_dict.index=="OR8J1",0]='ENSG00000172487'
har2000_dict.loc[har2000_dict.index=="LRP6",0]='ENSG00000070018'
har2000_dict.loc[har2000_dict.index=="MANSC1",0]='ENSG00000111261'
har2000_dict.loc[har2000_dict.index=="DUSP16",0]='ENSG00000111266'
har2000_dict.loc[har2000_dict.index=="PCDH20",0]='ENSG00000280165'
har2000_dict.loc[har2000_dict.index=="DHRS4L1",0]='ENSG00000285467'
har2000_dict.loc[har2000_dict.index=="HERC2P3",0]='ENSG00000180229'
har2000_dict.loc[har2000_dict.index=="GOLGA6L6",0]='ENSG00000277322'
har2000_dict.loc[har2000_dict.index=="GOLGA6L1",0]='ENSG00000273976'
har2000_dict.loc[har2000_dict.index=="TUBGCP5",0]='ENSG00000275835'
har2000_dict.loc[har2000_dict.index=="GOLGA8G",0]='ENSG00000183629'
har2000_dict.loc[har2000_dict.index=="ULK4P1",0]='ENSG00000261279'
har2000_dict.loc[har2000_dict.index=="FAN1",0]='ENSG00000198690'
har2000_dict.loc[har2000_dict.index=="NPIPA7",0]='ENSG00000214967'
har2000_dict.loc[har2000_dict.index=="XYLT1",0]='ENSG00000103489'
har2000_dict.loc[har2000_dict.index=="PKD1L2",0]='ENSG00000166473'
har2000_dict.loc[har2000_dict.index=="DOC2B",0]='ENSG00000272636'
har2000_dict.loc[har2000_dict.index=="CCDC144NL",0]='ENSG00000205212'
har2000_dict.loc[har2000_dict.index=="TBC1D3P2",0]='ENSG00000188755'
har2000_dict.loc[har2000_dict.index=="ZNF430",0]='ENSG00000118620'
har2000_dict.loc[har2000_dict.index=="ZNF100",0]='ENSG00000197020'
har2000_dict.loc[har2000_dict.index=="APOB",0]='ENSG00000084674'
har2000_dict.loc[har2000_dict.index=="GUSBP3",0]='ENSG00000253203'
har2000_dict.loc[har2000_dict.index=="PRMD9",0]='ENSG00000164256'
har2000_dict.loc[har2000_dict.index=="SERF1A",0]='ENSG00000172058'
har2000_dict.loc[har2000_dict.index=="OR2H2",0]='ENSG00000204657'
har2000_dict.loc[har2000_dict.index=="HLA-C",0]='ENSG00000204525'
har2000_dict.loc[har2000_dict.index=="SLC25A51P1",0]='ENSG00000220483'
har2000_dict.loc[har2000_dict.index=="IBTK",0]='ENSG00000005700'
har2000_dict.loc[har2000_dict.index=="PTPRK",0]='ENSG00000152894'
har2000_dict.loc[har2000_dict.index=="CRIM1",0]='ENSG00000150938'
har2000_dict.loc[har2000_dict.index=="DYNC1I2",0]='ENSG00000077380'
har2000_dict.loc[har2000_dict.index=="INO80D",0]='ENSG00000114933'
har2000_dict.loc[har2000_dict.index=="RBFOX2",0]='ENSG00000100320'
har2000_dict.loc[har2000_dict.index=="DLC1",0]='ENSG00000164741'
har2000_dict.loc[har2000_dict.index=="ZC3H3",0]='ENSG00000014164'
har2000_dict.loc[har2000_dict.index=="ACTR3BP2",0]='ENSG00000226481'
har2000_dict.loc[har2000_dict.index=="CHD4",0]='ENSG00000179242'
har2000_dict.loc[har2000_dict.index=="ABCC13",0]='ENSG00000243064'
har2000_dict.loc[har2000_dict.index=="CCT8",0]='ENSG00000156261'
har2000_dict.loc[har2000_dict.index=="COL6A4P1",0]='ENSG00000230524'
har2000_dict.loc[har2000_dict.index=="PLCL2",0]='ENSG00000154822'
har2000_dict.loc[har2000_dict.index=="KCNMB2",0]='ENSG00000197584'
har2000_dict.loc[har2000_dict.index=="RTP4",0]='ENSG00000136514'
har2000_dict.loc[har2000_dict.index=="GBA3",0]='ENSG00000249948'
har2000_dict.loc[har2000_dict.index=="CEP170P1",0]='ENSG00000154608'
har2000_dict.loc[har2000_dict.index=="CTSO",0]='ENSG00000256043'
har2000_dict.loc[har2000_dict.index=="GUSBP1",0]='ENSG00000183666'

In [66]:
har2000_dict=har2000_dict.rename(columns={0:'ens'})
har2000_dict

,ens
PARK7,ENSG00000116288
ERRFI1,ENSG00000116285
CAPZB,ENSG00000077549
LOC339505,None
LDLRAP1,ENSG00000157978
...,...
MAGEA11,ENSG00000185247
HSFX1,ENSG00000171116
LINC00894,None
MIR2114,ENSG00000252454


In [67]:
har2000_df=pd.DataFrame(data={'gene':har2000_dict.index,'ens':har2000_dict.ens})
har2000_df.index=range(2163)
har2000_df=har2000_df[~har2000_df.ens.isna()]
har2000_df

,gene,ens
0,PARK7,ENSG00000116288
1,ERRFI1,ENSG00000116285
2,CAPZB,ENSG00000077549
4,LDLRAP1,ENSG00000157978
6,TRNP1,ENSG00000253368
...,...,...
2157,AFF2,ENSG00000155966
2158,MAGEA11,ENSG00000185247
2159,HSFX1,ENSG00000171116
2161,MIR2114,ENSG00000252454


In [68]:
har2000_df.to_csv("/scratch200/reutj/data/hsg_gene_lists/har2000_df.csv")

<h2> HAR interactions paper </h2>

<b>here, i take the new article which examined C-hiC interactions of HARs and HGE in human and chimanzee neurons.</b> 
I will filter the interactions that were found for relevant interactions. then using bed intersection to find the genes that those interactions map to -up to 5kb from the annotated TSS .

In [75]:
#read the data
interctions_nsc=pd.read_csv("/scratch200/reutj/data/HAR_interactions_hNSCs.txt",sep="\t")

In [76]:
interctions_nsc

,bait_chr,bait_start,bait_end,bait_name,otherEnd_chr,otherEnd_start,otherEnd_end,CHiCAGO_score
0,chr1,2919723,2922344,2xHAR.521,chr1,2840312,2841269,4.61
1,chr1,2919723,2922344,2xHAR.521,chr1,2890866,2891461,5.01
2,chr1,2919723,2922344,2xHAR.521,chr1,3335258,3335985,5.27
3,chr1,3173000,3174883,"2xHAR.305,HAR98",chr1,3102995,3105012,5.75
4,chr1,3173000,3174883,"2xHAR.305,HAR98",chr1,3133885,3134162,5.04
...,...,...,...,...,...,...,...,...
39088,chrX,148702040,148703483,2xHAR.315,chrX,148662914,148663296,5.49
39089,chrX,148702040,148703483,2xHAR.315,chrX,148847608,148848377,5.54
39090,chrX,148702040,148703483,2xHAR.315,chrX,149052021,149052253,4.83
39091,chrX,148702040,148703483,2xHAR.315,chrX,149059323,149060527,4.85


In [77]:
interctions_neurons=pd.read_csv("/scratch200/reutj/data/HAR_interactions_human_neurons.txt",sep="\t")

In [91]:
interctions_neurons

,bait_chr,bait_start,bait_end,bait_name,otherEnd_chr,otherEnd_start,otherEnd_end,CHiCAGO_score
0,chr1,2919723,2922344,2xHAR.521,chr1,2651817,2654865,6.12
1,chr1,2919723,2922344,2xHAR.521,chr1,2939229,2940099,7.26
2,chr1,2919723,2922344,2xHAR.521,chr1,3295441,3296110,6.68
3,chr1,2919723,2922344,2xHAR.521,chr1,3387166,3389665,4.58
4,chr1,2919723,2922344,2xHAR.521,chr1,3412521,3415641,4.60
...,...,...,...,...,...,...,...,...
21410,chrX,148247240,148248402,2xHAR.139_tert,chrX,148338047,148339020,4.90
21411,chrX,148701831,148702036,2xHAR.315,chrX,148887657,148889198,4.66
21412,chrX,148701831,148702036,2xHAR.315,chrX,148994927,148995622,5.32
21413,chrX,148701831,148702036,2xHAR.315,chrX,149128209,149129008,5.42


In [78]:
import re

In [83]:
#filter interactions that are HGE-(DA) or contain sec and tert sequence.
filt_ind=[i is None for i in 
 list(map(lambda x: re.search("^DA", x) or re.search("_sec$", x) or re.search("_tert$", x),
          interctions_nsc.bait_name.tolist()))]

In [90]:
#now we will filter also intearction wih chichago 5 and above
print(len(interctions_nsc[filt_ind]))# keeps still most of interactions
len(interctions_nsc[(filt_ind) & (interctions_nsc.CHiCAGO_score>=5)]) #filters the amount to about third of the interactions..

18533


/tmp/ipykernel_173683/1862352438.py:3: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  len(interctions_nsc[(filt_ind) & (interctions_nsc.CHiCAGO_score>=5)])


11568

In [94]:
#apply the filtering
interctions_nsc=interctions_nsc[(filt_ind) & (interctions_nsc.CHiCAGO_score>=5)]

#do the same for neurons
filt_ind2=[i is None for i in 
 list(map(lambda x: re.search("^DA", x) or re.search("_sec$", x) or re.search("_tert$", x),
          interctions_neurons.bait_name.tolist()))]
print(len(interctions_neurons[filt_ind2]))# keeps still most of interactions
len(interctions_neurons[(filt_ind2) & (interctions_neurons.CHiCAGO_score>=5)]) #filters the amount to about quarter of the interactions..

interctions_neurons=interctions_neurons[(filt_ind2) & (interctions_neurons.CHiCAGO_score>=5)]

9523


/tmp/ipykernel_173683/1370342327.py:2: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  interctions_nsc=interctions_nsc[(filt_ind) & (interctions_nsc.CHiCAGO_score>=5)]
/tmp/ipykernel_173683/1370342327.py:9: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  len(interctions_neurons[(filt_ind2) & (interctions_neurons.CHiCAGO_score>=5)]) #filters the amount to about quarter of the interactions..
/tmp/ipykernel_173683/1370342327.py:11: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, In

In [95]:
#save to bed files..
#reorder columns to be chr,start,end,hi-c bait..
interactions_nsc=interctions_nsc.iloc[:,[4,5,6,3]]
interactions_neurons=interctions_neurons.iloc[:,[4,5,6,3]]

interactions_nsc.to_csv("/scratch200/reutj/data/interactions_nsc.bed", sep='\t', header=False, index=False)
interactions_neurons.to_csv("/scratch200/reutj/data/interactions_neurons.bed", sep='\t', header=False, index=False)

##  from bed genes intersection to complete list

In [98]:
#read file
interactions_har = pd.read_csv("/scratch200/reutj/data/neurons_har_genes_intersections.bed", sep='\t', header=None)


In [16]:
interactions_har

,0,1,2,3,4,5,6,7,8,9,10,11
0,chr1,2636985,2806693,ENSG00000215912.13,TTC34,protein_coding,-,chr1,2651817,2654865,2xHAR.521,3048
1,chr1,3064167,3438621,ENSG00000142611.17,PRDM16,protein_coding,+,chr1,3146386,3147536,"2xHAR.305,HAR98",1150
2,chr1,3064167,3438621,ENSG00000142611.17,PRDM16,protein_coding,+,chr1,3295441,3296110,2xHAR.521,669
3,chr1,6780453,7769706,ENSG00000171735.20,CAMTA1,protein_coding,+,chr1,7089190,7089850,HACNS512,660
4,chr1,6780453,7769706,ENSG00000171735.20,CAMTA1,protein_coding,+,chr1,7266306,7266850,HACNS512,544
...,...,...,...,...,...,...,...,...,...,...,...,...
4079,chrX,141929495,142740581,ENSG00000288098.1,ENSG00000288098,lncRNA,+,chrX,142057919,142058513,HACNS833,594
4080,chrX,141929495,142740581,ENSG00000288098.1,ENSG00000288098,lncRNA,+,chrX,142057919,142058513,HACNS833,594
4081,chrX,141929495,142740581,ENSG00000288098.1,ENSG00000288098,lncRNA,+,chrX,142421704,142422196,HACNS833,492
4082,chrX,148495616,149000663,ENSG00000155966.14,AFF2,protein_coding,+,chrX,148699012,148699869,2xHAR.315,857


In [99]:
interactions_har=interactions_har.rename(columns={0:'chr',1:'int_start',2:'int_end',3:'gene_ens',4:'gene_name',5:'annotated_as',
                          6:'strand',7:'har_chr',8:'har_start',9:'har_end',10:'har_name',11:'interaction_length'})
#remove hsub from hars and hars that still have DA in their name:
filt_ind=[i is None for i in list(map(lambda x: re.search("DA[0-9]", x) or re.search("^hSub", x),
          interactions_har.har_name.tolist()))]
interactions_har=interactions_har[filt_ind]

In [100]:
har_genes_new_neurons=interactions_har.groupby(['har_name','har_start','har_end'])[['gene_ens','gene_name','annotated_as']].agg(', '.join).reset_index()
har_genes_new_neurons

,har_name,har_start,har_end,gene_ens,gene_name,annotated_as
0,2xHAR.1,139127366,139128113,ENSG00000129682.16,FGF13,protein_coding
1,2xHAR.100,41445714,41447032,"ENSG00000165966.16, ENSG00000257228.5","PDZRN4, ENSG00000257228","protein_coding, lncRNA"
2,2xHAR.100,41497161,41497547,ENSG00000165966.16,PDZRN4,protein_coding
3,2xHAR.101,16320757,16322001,ENSG00000205549.10,LINC03041,lncRNA
4,2xHAR.101,16421501,16422074,ENSG00000173068.19,BNC2,protein_coding
...,...,...,...,...,...,...
3285,"hLinAR.835,human.sel",57414328,57414794,ENSG00000153944.12,MSI2,protein_coding
3286,"hLinAR.835,human.sel",57430694,57431584,ENSG00000153944.12,MSI2,protein_coding
3287,"hLinAR.835,human.sel",57450772,57451361,ENSG00000153944.12,MSI2,protein_coding
3288,"hLinAR.835,human.sel",57748530,57748930,ENSG00000166329.3,CCDC182,protein_coding


In [85]:
interactions_har2 = pd.read_csv("/scratch200/reutj/data/nsc_har_genes_intersections.bed", sep='\t', header=None,
                              names=['int_chr','int_start','int_end','gene_ens','gene_name','annotated_as','strand',
                          'har_chr','har_start','har_end','har_name','interaction_length'])

filt_ind=[i is None for i in list(map(lambda x: re.search("DA[0-9]", x) or re.search("^hSub", x),
          interactions_har2.har_name.tolist()))]
interactions_har2=interactions_har2[filt_ind]
interactions_har2

,int_chr,int_start,int_end,gene_ens,gene_name,annotated_as,strand,har_chr,har_start,har_end,har_name,interaction_length
0,chr1,3064167,3438621,ENSG00000142611.17,PRDM16,protein_coding,+,chr1,3102995,3105012,"2xHAR.305,HAR98",2017
1,chr1,3064167,3438621,ENSG00000142611.17,PRDM16,protein_coding,+,chr1,3133885,3134162,"2xHAR.305,HAR98",277
2,chr1,3064167,3438621,ENSG00000142611.17,PRDM16,protein_coding,+,chr1,3215283,3217196,"2xHAR.305,HAR98",1913
3,chr1,3064167,3438621,ENSG00000142611.17,PRDM16,protein_coding,+,chr1,3335258,3335985,2xHAR.521,727
4,chr1,3132926,3138709,ENSG00000226286.1,ENSG00000226286,lncRNA,-,chr1,3133885,3134162,"2xHAR.305,HAR98",277
...,...,...,...,...,...,...,...,...,...,...,...,...
9502,chrX,141929495,142740581,ENSG00000288098.1,ENSG00000288098,lncRNA,+,chrX,142149050,142149439,HACNS833,389
9503,chrX,141929495,142740581,ENSG00000288098.1,ENSG00000288098,lncRNA,+,chrX,142491904,142493125,HACNS833,1221
9504,chrX,142147861,142153866,ENSG00000236205.1,ENSG00000236205,unprocessed_pseudogene,-,chrX,142149050,142149439,HACNS833,389
9505,chrX,148495616,149000663,ENSG00000155966.14,AFF2,protein_coding,+,chrX,148662914,148663296,2xHAR.315,382


In [86]:
har_genes_new_nsc=interactions_har2.groupby(['har_name','har_start','har_end'])[['gene_ens','gene_name','annotated_as']].agg(', '.join).reset_index()
har_genes_new_nsc

,har_name,har_start,har_end,gene_ens,gene_name,annotated_as
0,2xHAR.100,42019614,42019945,ENSG00000286591.1,ENSG00000286591,lncRNA
1,2xHAR.101,16269365,16269713,ENSG00000205549.10,LINC03041,lncRNA
2,2xHAR.101,16414410,16415157,"ENSG00000205549.10, ENSG00000173068.19","LINC03041, BNC2","lncRNA, protein_coding"
3,2xHAR.101,16418755,16419184,ENSG00000173068.19,BNC2,protein_coding
4,2xHAR.101,16489060,16489536,ENSG00000173068.19,BNC2,protein_coding
...,...,...,...,...,...,...
7461,"hLinAR.835,human.sel",57291304,57291621,ENSG00000153944.12,MSI2,protein_coding
7462,"hLinAR.835,human.sel",57307667,57308119,ENSG00000153944.12,MSI2,protein_coding
7463,"hLinAR.835,human.sel",57425675,57426219,ENSG00000153944.12,MSI2,protein_coding
7464,"hLinAR.835,human.sel",57525018,57525749,"ENSG00000153944.12, ENSG00000266100.1","MSI2, ENSG00000266100","protein_coding, lncRNA"


In [89]:
#extract gene_names to create dataframe

ens,gene,annotated_as=[],[],[]
for har in range(len(har_genes_new_nsc)):
    ens.extend(i.split(".")[0].strip() for i in har_genes_new_nsc.gene_ens[har].split(","))
    gene.extend(i.split(".")[0].strip() for i in har_genes_new_nsc.gene_name[har].split(","))
    annotated_as.extend(i.split(".")[0].strip() for i in har_genes_new_nsc.annotated_as[har].split(","))


har_genes_nsc_df=pd.DataFrame({'ens':ens,'gene':gene,'annotated_as':annotated_as})
har_genes_nsc_df=har_genes_nsc_df.drop_duplicates()
har_genes_nsc_df

,ens,gene,annotated_as
0,ENSG00000286591,ENSG00000286591,lncRNA
1,ENSG00000205549,LINC03041,lncRNA
3,ENSG00000173068,BNC2,protein_coding
14,ENSG00000163377,TAFA4,protein_coding
17,ENSG00000048052,HDAC9,protein_coding
...,...,...,...
9123,ENSG00000258532,LINC02305,lncRNA
9124,ENSG00000224842,LMX1B-DT,lncRNA
9125,ENSG00000136944,LMX1B,protein_coding
9134,ENSG00000266100,ENSG00000266100,lncRNA


In [90]:
#extract gene_names to create dataframe

ens,gene,annotated_as=[],[],[]
for har in range(len(har_genes_new_neurons)):
    ens.extend(i.split(".")[0].strip() for i in har_genes_new_neurons.gene_ens[har].split(","))
    gene.extend(i.split(".")[0].strip() for i in har_genes_new_neurons.gene_name[har].split(","))
    annotated_as.extend(i.split(".")[0].strip() for i in har_genes_new_neurons.annotated_as[har].split(","))


har_genes_neurons_df=pd.DataFrame({'ens':ens,'gene':gene,'annotated_as':annotated_as})
har_genes_neurons_df=har_genes_neurons_df.drop_duplicates()
har_genes_neurons_df

,ens,gene,annotated_as
0,ENSG00000129682,FGF13,protein_coding
1,ENSG00000125637,PSD4,protein_coding
3,ENSG00000189223,PAX8-AS1,lncRNA
5,ENSG00000125618,PAX8,protein_coding
6,ENSG00000165966,PDZRN4,protein_coding
...,...,...,...
4075,ENSG00000259471,LINC01169,lncRNA
4076,ENSG00000137834,SMAD6,protein_coding
4081,ENSG00000171467,ZNF318,protein_coding
4082,ENSG00000130720,FIBCD1,protein_coding


In [101]:
#save to siles..
har_genes_neurons_df.to_csv("/scratch200/reutj/data/hsg_gene_lists/har_genes_neurons_df.csv")
har_genes_nsc_df.to_csv("/scratch200/reutj/data/hsg_gene_lists/har_genes_nsc_df.csv")
interactions_har.to_csv("/scratch200/reutj/data/hsg_gene_lists/har_interactions_mapped_neurons.csv")
interactions_har2.to_csv("/scratch200/reutj/data/hsg_gene_lists/har_interactions_mapped_nsc.csv")

In [ ]:
#save to gmt

In [108]:
har_genes_nsc_df=pd.read_csv("/scratch200/reutj/data/hsg_gene_lists/har_genes_nsc_df.csv")

In [110]:
output_file="/scratch200/reutj/data/hsg_gene_lists/har_genes_nsc_df.gmt"
genes=har_genes_nsc_df.gene.tolist()
with open(output_file, 'w') as f:
    # Create a line in GMT format for one gene set
    line = f"{har_genes_nsc_df}\tDescription\t{'\t'.join(genes)}\n"
    f.write(line)

In [118]:
har_genes_neurons_df=pd.read_csv("/scratch200/reutj/data/hsg_gene_lists/har_genes_neurons_df.csv")

In [119]:
output_file="/scratch200/reutj/data/hsg_gene_lists/har_genes_neurons_df.gmt"
genes=har_genes_neurons_df.gene.tolist()
with open(output_file, 'w') as f:
    # Create a line in GMT format for one gene set
    line = f"{har_genes_neurons_df}\tDescription\t{'\t'.join(genes)}\n"
    f.write(line)

In [116]:
updated_hsg_list=pd.read_csv("/scratch200/reutj/data/hsg_gene_lists/updated_hsg_list.csv")

In [117]:
output_file="/scratch200/reutj/data/hsg_gene_lists/updated_hsg_list.gmt"
genes=updated_hsg_list.gene.tolist()
with open(output_file, 'w') as f:
    # Create a line in GMT format for one gene set
    line = f"{updated_hsg_list}\tDescription\t{'\t'.join(genes)}\n"
    f.write(line)

In [120]:
ges_for_example=pd.read_csv("/scratch200/reutj/data/spec_score_tables/ges_spec_v3_Neuroblast.csv")

In [128]:
ges_for_example=ges_for_example.sort_values("ges_score", ascending=False)

In [129]:
ges_for_example[(ges_for_example.p_values<0.05)]

,Unnamed: 0,gene,ges_score,mean_expression_target,mean_expression_other,per_expressed_target,per_expressed_other,per_exp_Radial glia,per_exp_Neuronal IPC,per_exp_Neuron,per_exp_Glioblast,p_values,adj_p_val
980,980,LINC02058,16.155042,0.049053,0.004056,0.081112,0.009574,0.002605,0.047210,0.004792,0.004533,0.000,0.000000
1313,1313,AP005328.1,11.769171,0.029509,0.003349,0.050032,0.006920,0.001786,0.014802,0.010709,0.001445,0.000,0.000000
1244,1244,AP000894.3,11.682298,0.034555,0.003951,0.067532,0.009722,0.001265,0.020759,0.016598,0.000723,0.000,0.000000
1242,1242,EBF1,11.320093,0.074985,0.008848,0.087767,0.018796,0.018271,0.068232,0.007447,0.008475,0.000,0.000000
466,466,RELN,11.070372,0.257905,0.031120,0.253944,0.060541,0.032709,0.119033,0.081026,0.019183,0.000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9379,9379,MDH2,1.345415,0.614494,0.610095,0.808216,0.822687,0.850966,0.856004,0.819560,0.754894,0.032,0.083755
595,595,CHCHD2,1.342886,1.388971,1.381625,0.989426,0.985662,0.989469,0.994307,0.982883,0.978781,0.004,0.010890
7179,7179,RPL19,1.341256,2.267136,2.257886,0.999189,0.998412,0.998958,0.998599,0.998124,0.997963,0.002,0.005504
8721,8721,RPS21,1.340846,2.107955,2.099996,0.999122,0.996529,0.995981,0.999124,0.998874,0.990212,0.006,0.016252


In [132]:
ges_for_example2=pd.read_csv("/scratch200/reutj/data/spec_score_tables/ges_spec_v3_updated_Neuron.csv")

In [133]:
ges_for_example2=ges_for_example2.sort_values("ges_score", ascending=False)

In [135]:
ges_for_example2[(ges_for_example2.p_values<0.05)]

,Unnamed: 0,gene,ges_score,mean_expression_target,mean_expression_other,per_expressed_target,per_expressed_other,per_exp_Neuroblast,per_exp_Radial glia,per_exp_Neuronal IPC,per_exp_Glioblast,p_values,adj_p_val,exp_diff_all
0,0,SLC5A8,117.363193,0.043711,0.000528,0.091793,0.001299,0.001655,0.000968,0.001401,0.001117,0.000,0.000000,True
1,1,KRT31,106.686763,0.029479,0.000391,0.071241,0.000878,0.001115,0.000223,0.001226,0.001314,0.000,0.000000,True
2,2,GPR22,76.101683,0.106956,0.001991,0.195970,0.004620,0.007601,0.002047,0.004817,0.003219,0.000,0.000000,True
3,3,SLC26A4-AS1,66.109762,0.032795,0.000703,0.085991,0.001733,0.002399,0.000856,0.002190,0.001642,0.000,0.000000,True
4,4,GRIK3,47.610674,0.439843,0.013089,0.455936,0.026217,0.044188,0.009564,0.029517,0.018197,0.000,0.000000,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5843,5843,RPL24,1.426751,2.164838,2.149766,0.996969,0.998195,0.998818,0.998363,0.998686,0.996321,0.000,0.000000,False
5846,5846,ARPC3,1.426649,0.744158,0.739030,0.900846,0.883496,0.878551,0.883117,0.919769,0.866575,0.010,0.021115,False
5854,5854,TRIM2,1.425838,1.078666,1.071842,0.959934,0.919784,0.933854,0.864585,0.947359,0.969189,0.020,0.041929,False
5866,5866,RPL32,1.425225,2.602568,2.587216,0.998412,0.999422,0.999257,0.999702,0.999562,0.999146,0.000,0.000000,False


In [122]:
"/scratch200/reutj/data/spec_score_tables/ges_spec_v3_Neuroblast.csv".split("/")[-1]

'ges_spec_v3_Neuroblast.csv'

In [136]:
adata_cortex=sc.read_h5ad("/scratch200/reutj/data/updated_cortex_data_hg38.h5ad")

In [137]:
adata_cortex.obs.CellCycle

,Age,Agetext,Ageunit,All_fc_analysis_id,Analysis,CellCycle,CellCycle_G1,CellCycle_G2M,CellCycle_S,Cellconc,...,classes,mark,CellID,CellCycleFraction,Cycling,CellCyclePhase,CellClass,proliferation,CellCycleStatus,NPCs
CellID,,,,,,,,,,,,,,,,,,,,,
10X298_1:TGTGATGGTCCAGCAC,5.5,5.5w,pcw,4624,Queueing,0.002037,0.000543,0.001358,0.000136,1240,...,Neuroblast,0,10X298_1:TGTGATGGTCCAGCAC,0.001664,False,Non-cycling,Neuroblast,differentiating,differentiating_non_cycling,other
10X298_1:ATGCATGGTTAAAGTG,5.5,5.5w,pcw,4624,Queueing,0.002892,0.001285,0.001526,0.000080,1240,...,Neuroblast,0,10X298_1:ATGCATGGTTAAAGTG,0.002593,False,Non-cycling,Radial glia,proliferating,proliferating_non_cycling,NPCs
10X298_1:TTGGGCGTCACGTAGT,5.5,5.5w,pcw,4624,Queueing,0.002670,0.000191,0.002384,0.000095,1240,...,Neuroblast,0,10X298_1:TTGGGCGTCACGTAGT,0.002339,False,Non-cycling,Neuroblast,differentiating,differentiating_non_cycling,other
10X298_1:TACGCTCTCCCGAGGT,5.5,5.5w,pcw,4624,Queueing,0.003317,0.001167,0.001659,0.000491,1240,...,Neuroblast,0,10X298_1:TACGCTCTCCCGAGGT,0.003122,False,Non-cycling,Neuroblast,differentiating,differentiating_non_cycling,other
10X298_1:TTGCCTGCAAGTTGGG,5.5,5.5w,pcw,4624,Queueing,0.004388,0.000768,0.003510,0.000110,1240,...,Neuroblast,0,10X298_1:TTGCCTGCAAGTTGGG,0.003877,False,Non-cycling,Neuroblast,differentiating,differentiating_non_cycling,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10X257_2:TAACCAGCAGTAACAA,14.0,14w,pcw,3778,Queueing,0.025884,0.006604,0.010865,0.008415,770,...,Radial glia,0,10X257_2:TAACCAGCAGTAACAA,0.022677,True,S,Glioblast,proliferating,SG2M,other
10X257_1:GGATCTAAGTGGCAGT,14.0,14w,pcw,3612,Queueing,0.040516,0.002734,0.028880,0.008902,770,...,Radial glia,0,10X257_1:GGATCTAAGTGGCAGT,0.035481,True,S,Radial glia,proliferating,SG2M,NPCs
10X257_2:GTCAAACTCCTTGACC,14.0,14w,pcw,3778,Queueing,0.021243,0.007022,0.008866,0.005355,770,...,Radial glia,0,10X257_2:GTCAAACTCCTTGACC,0.019012,True,S,Glioblast,proliferating,SG2M,other


In [144]:
adata_cortex.obs.CellClass.unique()

CellID
10X298_1:TGTGATGGTCCAGCAC     Neuroblast
10X298_1:ATGCATGGTTAAAGTG    Radial glia
10X298_1:TTGGGCGTCACGTAGT     Neuroblast
10X298_1:TACGCTCTCCCGAGGT     Neuroblast
10X298_1:TTGCCTGCAAGTTGGG     Neuroblast
                                ...     
10X257_2:TAACCAGCAGTAACAA      Glioblast
10X257_1:GGATCTAAGTGGCAGT    Radial glia
10X257_2:GTCAAACTCCTTGACC      Glioblast
10X257_1:CCTAAGAGTCAGATTC    Radial glia
10X257_1:AATGGAATCGTGTGAT    Radial glia
Name: CellClass, Length: 297927, dtype: category
Categories (5, object): ['Glioblast', 'Neuroblast', 'Neuron', 'Neuronal IPC', 'Radial glia']

In [141]:
#add non cycling saparation in cellcycle status
adata_cortex.obs.CellCyclePhase=adata_cortex.obs.CellCyclePhase.astype('str')
adata_cortex.obs.loc[(adata_cortex.obs.proliferation=="proliferating") & (adata_cortex.obs.CellCyclePhase =="Non-cycling"),'CellCyclePhase']="proliferating_non_cycling"
adata_cortex.obs.loc[(adata_cortex.obs.proliferation!="proliferating") & (adata_cortex.obs.CellCyclePhase =="Non-cycling"),'CellCyclePhase']="differentiating_non_cycling"
adata_cortex.obs.CellCyclePhase=adata_cortex.obs.CellCyclePhase.astype('category')
adata_cortex.obs.CellCyclePhase

CellID
10X298_1:TGTGATGGTCCAGCAC    differentiating_non_cycling
10X298_1:ATGCATGGTTAAAGTG      proliferating_non_cycling
10X298_1:TTGGGCGTCACGTAGT    differentiating_non_cycling
10X298_1:TACGCTCTCCCGAGGT    differentiating_non_cycling
10X298_1:TTGCCTGCAAGTTGGG    differentiating_non_cycling
                                        ...             
10X257_2:TAACCAGCAGTAACAA                              S
10X257_1:GGATCTAAGTGGCAGT                              S
10X257_2:GTCAAACTCCTTGACC                              S
10X257_1:CCTAAGAGTCAGATTC                             G1
10X257_1:AATGGAATCGTGTGAT                          PostM
Name: CellCyclePhase, Length: 297927, dtype: category
Categories (6, object): ['G1', 'G2M', 'PostM', 'S', 'differentiating_non_cycling', 'proliferating_non_cycling']

In [143]:
adata_cortex.write_h5ad("/scratch200/reutj/data/updated_cortex_data_hg38.h5ad")

In [145]:
sc.pp.filter_cells(adata_cortex, min_genes=200)
sc.pp.filter_genes(adata_cortex, min_cells=100)

In [ ]:
gene = ["Apples", "Oranges", "Pears", "Bananas", "Pineapples"]

with open("genes_above_100.txt", mode="w") as fruitfile:
    fruitfile.write("\n".join(fruits) + "\n")
